In [2]:
! pip install google-generativeai


  Using cached google_generativeai-0.8.6-py3-none-any.whl.metadata (3.9 kB)
Using cached google_generativeai-0.8.6-py3-none-any.whl (155 kB)



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import streamlit as st
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import os
import google.generativeai as genai
print("Gemini library loaded successfully")


Gemini library loaded successfully


C:\Users\Admin\AppData\Local\Temp\ipykernel_6656\3313947215.py:8: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [4]:
! pip install google-genai



   ---------------------------------------- 0/2 [websockets]
   ---------------------------------------- 0/2 [websockets]
   ---------------------------------------- 0/2 [websockets]
   ---------------------------------------- 0/2 [websockets]
   ---------------------------------------- 0/2 [websockets]
   ---------------------------------------- 0/2 [websockets]
   ---------------------------------------- 0/2 [websockets]
   ---------------------------------------- 0/2 [websockets]
   ---------------------------------------- 0/2 [websockets]
   ---------------------------------------- 0/2 [websockets]
   ---------------------------------------- 0/2 [websockets]
   ---------------------------------------- 0/2 [websockets]
   ---------------------------------------- 0/2 [websockets]
   ---------------------------------------- 0/2 [websockets]
   -------------------- ------------------- 1/2 [google-genai]
   -------------------- ------------------- 1/2 [google-genai]
   ----------------


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import google.genai as genai


In [6]:

st.set_page_config(page_title="AI-NIDS Student Project", layout="wide")

st.title("AI-Based Network Intrusion Detection System")
st.markdown("""
**Student Project**: This system uses **Random Forest** to detect Network attacks and **Google Gemini** to explain the packets.
""")



2026-01-06 19:37:15.798 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-06 19:37:15.799 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-06 19:37:16.772 
  command:

    streamlit run C:\Python\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-01-06 19:37:16.773 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-06 19:37:16.774 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-06 19:37:16.776 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-06 19:37:16.777 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mod

DeltaGenerator()

In [18]:
# --- CONFIGURATION ---
DATA_FILE = "https://huggingface.co/spaces/ModelMuse02/NIDS_Project/resolve/main/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv"



In [19]:
# --- SIDEBAR: SETTINGS ---
st.sidebar.header("1. Settings")
# CHANGED: API Key label for Gemini
gemini_api_key = st.sidebar.text_input("Gemini API Key", type="password")
st.sidebar.caption("[Get a free Google API key here](https://aistudio.google.com/app/apikey)")

st.sidebar.header("2. Model Training")

@st.cache_data
def load_data(filepath):
    try:
        # Load small chunk for performance
        df = pd.read_csv(filepath, nrows=15000)
        df.columns = df.columns.str.strip()
        df.replace([np.inf, -np.inf], np.nan, inplace=True)
        df.dropna(inplace=True)
        return df
    except FileNotFoundError:
        return None

def train_model(df):
    features = ['Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 
                'Total Length of Fwd Packets', 'Fwd Packet Length Max', 
                'Flow IAT Mean', 'Flow IAT Std', 'Flow Packets/s']
    target = 'Label'
    
    missing_cols = [c for c in features if c not in df.columns]
    if missing_cols:
        st.error(f"Missing columns in CSV: {missing_cols}")
        return None, 0, [], None, None

    X = df[features]
    y = df[target]
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
    
    clf = RandomForestClassifier(n_estimators=10, max_depth=10, random_state=42)
    clf.fit(X_train, y_train)
    
    score = accuracy_score(y_test, clf.predict(X_test))
    return clf, score, features, X_test, y_test



2026-01-06 19:42:46.774 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-06 19:42:46.775 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-06 19:42:46.776 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-06 19:42:46.777 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-06 19:42:46.778 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-06 19:42:46.779 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-06 19:42:46.780 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-06 19:42:46.782 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [20]:
# --- APP LOGIC ---
df = load_data(DATA_FILE)

if df is None:
    st.error(f"Error: File '{DATA_FILE}' not found. Please ensure the CSV is in the same folder.")
    st.stop()

st.sidebar.success(f"Dataset Loaded: {len(df)} rows")

if st.sidebar.button("Train Model Now"):
    with st.spinner("Training model..."):
        clf, accuracy, feature_names, X_test, y_test = train_model(df)
        if clf:
            st.session_state['model'] = clf
            st.session_state['features'] = feature_names
            st.session_state['X_test'] = X_test 
            st.session_state['y_test'] = y_test
            st.sidebar.success(f"Training Complete! Accuracy: {accuracy:.2%}")

st.header("3. Threat Analysis Dashboard")

if 'model' in st.session_state:
    col1, col2 = st.columns(2)
    
    with col1:
        st.subheader("Simulation")
        st.info("Pick a random packet from the test data to simulate live traffic.")
        
        if st.button("🎲 Capture Random Packet"):
            random_idx = np.random.randint(0, len(st.session_state['X_test']))
            packet_data = st.session_state['X_test'].iloc[random_idx]
            actual_label = st.session_state['y_test'].iloc[random_idx]
            
            st.session_state['current_packet'] = packet_data
            st.session_state['actual_label'] = actual_label
            
    if 'current_packet' in st.session_state:
        packet = st.session_state['current_packet']
        
        with col1:
            st.write("**Packet Header Info:**")
            st.dataframe(packet, use_container_width=True)

        with col2:
            st.subheader("AI Detection Result")
            prediction = st.session_state['model'].predict([packet])[0]
            
            if prediction == "BENIGN":
                st.success(f" STATUS: **SAFE (BENIGN)**")
            else:
                st.error(f"🚨 STATUS: **ATTACK DETECTED ({prediction})**")
            
            st.caption(f"Ground Truth Label: {st.session_state['actual_label']}")

            st.markdown("---")
            st.subheader(" Ask AI Analyst (Gemini)")
            
            if st.button("Generate Explanation"):
                if not gemini_api_key:
                    st.warning("Please enter your Gemini API Key in the sidebar first.")
                else:
                    try:
                        #  Gemini configuration and model call
                        genai.configure(api_key=gemini_api_key)
                        model = genai.GenerativeModel('gemini-1.5-flash')
                        
                        prompt = f"""
                        You are a cybersecurity analyst. 
                        A network packet was detected as: {prediction}.
                        
                        Packet Technical Details:
                        {packet.to_string()}
                        
                        Please explain:
                        1. Why these specific values (like Flow Duration or Packet Length) might indicate {prediction}.
                        2. If it is BENIGN, explain why it looks normal.
                        3. Keep the answer short and simple for a student.
                        """

                        with st.spinner("Gemini is analyzing the packet..."):
                            response = model.generate_content(prompt)
                            st.info(response.text)
                            
                    except Exception as e:
                        st.error(f"API Error: {e}")
else:
    st.info(" Waiting for model training. Click **'Train Model Now'** in the sidebar.")

2026-01-06 19:42:48.984 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-06 19:42:48.985 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-06 19:42:48.987 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-06 19:42:48.988 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-06 19:42:49.522 Thread 'Thread-10': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-06 19:42:49.531 Thread 'Thread-10': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-06 19:42:49.534 Thread 'Thread-10': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-01-06 19:44:58.104 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare m

In [12]:
import os

# Add Git to the PATH for this session
os.environ['PATH'] += r";C:\Program Files\Git\cmd"

# Verify if Git is now recognized
!git --version


git version 2.51.0.windows.1


In [9]:
!git init

Initialized empty Git repository in C:/Users/Admin/Projects/Projects/Project_6/.git/


In [10]:
!git add .

In [11]:
!git commit -m "first commit"

[master (root-commit) 275de40] first commit
 3 files changed, 475 insertions(+)
 create mode 100644 .ipynb_checkpoints/NIDS_Development-checkpoint.ipynb
 create mode 100644 NIDS_Development.ipynb
 create mode 100644 app.py


In [12]:
!git branch -M main

In [13]:
!git remote add origin https://github.com/igituser007/AI_NIDS_Project.git

In [16]:
!git push -u origin main

branch 'main' set up to track 'origin/main'.


To https://github.com/igituser007/AI_NIDS_Project.git
   33cda15..41452cb  main -> main
